# Training pipeline: will a driver score points?

**Business question.** A fantasy-F1 platform wants to recommend drivers who will finish in the points (top 10). The model runs **after qualifying and before the race**, so it may only use information that exists at that moment.

**What this notebook does**
1. Loads seven F1 tables and builds one row per *driver per race*.
2. Builds pre-race features (no leakage) and a chronological train / validation / test split.
3. Trains the **same logistic regression three ways**: scikit-learn, a manual PyTorch loop, and `nn.Module` + `torch.optim`.
4. Compares them against a naive baseline and saves every artifact the Gradio app needs.

In [ ]:
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, log_loss, precision_score,
                             recall_score, roc_auc_score)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Configuration (YOUR DECISION)

Every choice that comes from the business framing lives here, so nothing is hidden inside the code.

- **Years:** only the hybrid-engine era (2014 onward). Older cars and rules are not comparable, and it keeps the data small.
- **Split:** train 2014-2019, validation 2020-2021, test 2022-2024. It is chronological because a random split would leak future races into training.
- **Costs:** a false positive (recommending a driver who does not score) costs 1; a false negative (missing a driver who scores) costs 2. They set the decision threshold.
- **Windows:** how many past races define "recent form" and "reliability".

In [ ]:
START_YEAR, END_YEAR = 2014, 2024
TRAIN_END, VAL_END = 2019, 2021          # test = seasons after VAL_END

FORM_WINDOW = 5            # past races used for recent form
RELIABILITY_WINDOW = 10    # past races used for retirement rate
ROOKIE_RACES = 20          # fewer career starts than this = rookie

COST_FP, COST_FN = 1, 2    # cost of each error type

C_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]   # regularisation strengths to try
EPOCHS, LR = 3000, 0.5                      # for both PyTorch versions

DATA_DIR = Path("F1 Data") if Path("F1 Data").exists() else Path("data")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Reading data from:", DATA_DIR)

## 2. Load the data

The dataset is relational: seven tables linked by ids. Ergast writes missing values as `\N`, so we tell pandas to read those as `NaN`.

In [ ]:
def read(name):
    return pd.read_csv(DATA_DIR / f"{name}.csv", na_values=["\\N"])

results      = read("results")
races        = read("races")
drivers      = read("drivers")
constructors = read("constructors")
circuits     = read("circuits")
qualifying   = read("qualifying")
status       = read("status")

for name, t in [("results", results), ("races", races), ("drivers", drivers),
                ("constructors", constructors), ("circuits", circuits),
                ("qualifying", qualifying), ("status", status)]:
    print(f"{name:13s} {t.shape}")

## 3. Base table and target

`results` is the spine: one row per driver per race. We join the other tables onto it and sort **by date**, which every "past races" feature relies on.

The target is `scores_points = 1` when `points > 0`. We also create two helper columns used *only to build history features* (`top10`, `dnf`); they describe the race itself, so they must never be model inputs for that same race.

**Leakage rule:** after the race we know `position`, `points`, `laps`, `time`, `status`... none of these can be a feature. The list below is checked by an assertion later.

In [ ]:
df = (results
      .merge(races[["raceId", "year", "round", "date", "circuitId", "name"]]
             .rename(columns={"name": "race_name"}), on="raceId")
      .merge(drivers[["driverId", "forename", "surname", "dob"]], on="driverId")
      .merge(constructors[["constructorId", "name"]]
             .rename(columns={"name": "team"}), on="constructorId")
      .merge(status, on="statusId")
      .merge(circuits[["circuitId", "alt"]], on="circuitId"))

df["date"] = pd.to_datetime(df["date"])
df["dob"] = pd.to_datetime(df["dob"])
df["driver"] = df["forename"] + " " + df["surname"]
df = df.sort_values(["date", "raceId", "positionOrder"]).reset_index(drop=True)

# target
df["scores_points"] = (df["points"] > 0).astype(int)

# helpers for history features (describe the race itself -> never used as inputs)
df["top10"] = (df["positionOrder"] <= 10).astype(int)
finished = df["status"].eq("Finished") | df["status"].str.startswith("+")
df["dnf"] = (~finished).astype(int)

# columns that only exist AFTER the race
LEAKY = ["position", "positionText", "positionOrder", "points", "laps", "time",
         "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed",
         "statusId", "status", "scores_points", "top10", "dnf"]

print(df.shape)

## 4. Pre-race features: grid and qualifying

- **`quali_pos`**: position in qualifying.
- **`q1_gap`**: Q1 lap time relative to the fastest Q1 time (0.01 = 1% slower than pole pace). Everyone takes part in Q1, so it exists for the whole field.
- **`grid`**: starting position. A value of `0` means the driver started from the pit lane, so we recode it to *last place + 1* and add a `pitlane` flag; leaving 0 would make them look like pole sitters.
- **`grid_vs_mate`**: grid minus the teammate's grid. It separates the driver from the car.

In [ ]:
def to_seconds(t):
    if pd.isna(t):
        return np.nan
    minutes, seconds = t.split(":")
    return int(minutes) * 60 + float(seconds)

qualifying = qualifying.copy()
qualifying["q1_sec"] = qualifying["q1"].map(to_seconds)
qualifying["q1_gap"] = qualifying["q1_sec"] / qualifying.groupby("raceId")["q1_sec"].transform("min") - 1

df = df.merge(
    qualifying[["raceId", "driverId", "position", "q1_gap"]].rename(columns={"position": "quali_pos"}),
    on=["raceId", "driverId"], how="left")

# pit-lane starts: grid 0 -> last place + 1
df["pitlane"] = (df["grid"] == 0).astype(int)
df.loc[df["grid"] == 0, "grid"] = df.groupby("raceId")["grid"].transform("max") + 1

# grid relative to teammate(s) in the same race
grp = df.groupby(["raceId", "constructorId"])["grid"]
n_cars = grp.transform("count")
mate_mean = (grp.transform("sum") - df["grid"]) / (n_cars - 1).clip(lower=1)
df["grid_vs_mate"] = np.where(n_cars > 1, df["grid"] - mate_mean, 0.0)

## 5. History features (no leakage)

Recent form and reliability are averages over **previous** races only. The key is `.shift(1)`: it moves each value one race forward, so the current race is never included in its own feature.

- **`drv_top10_form` / `team_top10_form`**: share of top-10 finishes in the last 5 races (driver and constructor). We use a top-10 indicator instead of raw points because the points system changed over the years.
- **`drv_dnf_rate` / `team_dnf_rate`**: share of races not finished in the last 10 (reliability).

History is computed on the *full* table (before filtering to 2014+) so the first races of 2014 already have a past to look at.

In [ ]:
def lagged_mean(series, window, min_periods):
    return series.shift(1).rolling(window, min_periods=min_periods).mean()

# driver level
df["drv_top10_form"] = df.groupby("driverId")["top10"].transform(
    lambda s: lagged_mean(s, FORM_WINDOW, 3))
df["drv_dnf_rate"] = df.groupby("driverId")["dnf"].transform(
    lambda s: lagged_mean(s, RELIABILITY_WINDOW, 3))

# constructor level: average over the team's cars in each race, then lag
team = (df.groupby(["constructorId", "raceId"], as_index=False)
          .agg(date=("date", "first"), t_top10=("top10", "mean"), t_dnf=("dnf", "mean"))
          .sort_values("date"))
team["team_top10_form"] = team.groupby("constructorId")["t_top10"].transform(
    lambda s: lagged_mean(s, FORM_WINDOW, 2))
team["team_dnf_rate"] = team.groupby("constructorId")["t_dnf"].transform(
    lambda s: lagged_mean(s, RELIABILITY_WINDOW, 3))

df = df.merge(team[["constructorId", "raceId", "team_top10_form", "team_dnf_rate"]],
              on=["constructorId", "raceId"], how="left")

## 6. Driver profile and final table

- **`age`** at the race date and **`experience`** (career races before this one; the counter starts at 0). **`rookie`** flags drivers with fewer than 20 starts.
- **`alt`**: circuit altitude.

Then we keep only 2014-2024 and check how much is missing. Missing values come mostly from rookies without history and from rare gaps in qualifying data.

In [ ]:
df["age"] = (df["date"] - df["dob"]).dt.days / 365.25
df["experience"] = df.groupby("driverId").cumcount()
df["rookie"] = (df["experience"] < ROOKIE_RACES).astype(int)

FEATURES = ["grid", "pitlane", "quali_pos", "q1_gap", "grid_vs_mate",
            "drv_top10_form", "team_top10_form", "drv_dnf_rate", "team_dnf_rate",
            "age", "experience", "rookie", "alt"]

# safety check: no post-race column may be a feature
assert not set(FEATURES) & set(LEAKY), "Leakage: a post-race column is used as a feature"

data = df[df["year"].between(START_YEAR, END_YEAR)].reset_index(drop=True)
print("rows:", len(data), "| features:", len(FEATURES))
print("\nshare of missing values per feature:")
print(data[FEATURES].isna().mean().round(3).to_string())

## 7. Chronological split

Training uses the earliest seasons, validation the next ones (to choose the regularisation and the threshold), and test the latest seasons (touched only for the final comparison). We assert that the periods do not overlap in time.

In [ ]:
data["split"] = np.where(data["year"] <= TRAIN_END, "train",
                 np.where(data["year"] <= VAL_END, "val", "test"))

train = data[data["split"] == "train"]
val   = data[data["split"] == "val"]
test  = data[data["split"] == "test"]

assert train["date"].max() < val["date"].min() <= val["date"].max() < test["date"].min()

summary = data.groupby("split").agg(rows=("scores_points", "size"),
                                    positive_rate=("scores_points", "mean"),
                                    first_year=("year", "min"), last_year=("year", "max"))
print(summary.loc[["train", "val", "test"]].round(3).to_string())

## 8. Imputation and scaling

Missing values are filled with the **training median**; features are standardised with the **training mean and standard deviation**. These statistics come from the training set only, otherwise information from validation or test would leak in. We save them so the app can apply the identical transformation later.

In [ ]:
median = train[FEATURES].median()
filled_train = train[FEATURES].fillna(median)
mean = filled_train.mean()
std = filled_train.std().replace(0, 1.0)

def transform(frame):
    return ((frame[FEATURES].fillna(median) - mean) / std).values.astype(np.float32)

X_train, X_val, X_test = transform(train), transform(val), transform(test)
y_train = train["scores_points"].values
y_val   = val["scores_points"].values
y_test  = test["scores_points"].values
print(X_train.shape, X_val.shape, X_test.shape)

## 9. Metrics, cost, and the naive baseline

The decision threshold is chosen to minimise the **expected cost per driver-race** with the costs set above: `(FP * COST_FP + FN * COST_FN) / n`. We pick it on the **validation** set and report it on the test set.

**Baseline:** "a driver scores if he starts in the top 10". Any trained model has to beat this simple rule.

In [ ]:
def expected_cost(y, p, thr):
    pred = p >= thr
    fp = ((pred == 1) & (y == 0)).sum()
    fn = ((pred == 0) & (y == 1)).sum()
    return (fp * COST_FP + fn * COST_FN) / len(y)

def best_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    costs = [expected_cost(y, p, t) for t in grid]
    return float(grid[int(np.argmin(costs))])

def evaluate(name, p_val, p_test):
    thr = best_threshold(y_val, p_val)
    pred = p_test >= thr
    return {
        "model": name,
        "threshold": round(thr, 2),
        "auc": round(roc_auc_score(y_test, p_test), 4),
        "log_loss": round(log_loss(y_test, p_test), 4),
        "accuracy@0.5": round(accuracy_score(y_test, p_test >= 0.5), 4),
        "recall@thr": round(recall_score(y_test, pred), 4),
        "precision@thr": round(precision_score(y_test, pred), 4),
        "cost@0.5": round(expected_cost(y_test, p_test, 0.5), 4),
        "cost@thr": round(expected_cost(y_test, p_test, thr), 4),
    }

# naive baseline: scores if starting in the top 10
base_test = test["grid"].between(1, 10).astype(float).values
baseline = {
    "model": "baseline (grid <= 10)", "threshold": 0.5,
    "auc": round(roc_auc_score(y_test, -test["grid"].values), 4), "log_loss": np.nan,
    "accuracy@0.5": round(accuracy_score(y_test, base_test), 4),
    "recall@thr": round(recall_score(y_test, base_test), 4),
    "precision@thr": round(precision_score(y_test, base_test), 4),
    "cost@0.5": round(expected_cost(y_test, base_test, 0.5), 4),
    "cost@thr": round(expected_cost(y_test, base_test, 0.5), 4),
}
print(pd.Series(baseline).to_string())

## 10. Model 1: scikit-learn logistic regression

Logistic regression models the **probability** of scoring: `p = sigmoid(w . x + b)`. We use it because the target is binary and we need a probability to apply a cost-based threshold.

Regularisation (`C`; smaller = stronger L2 penalty) is the only thing we tune, using **validation log loss**. The two PyTorch models reuse this strength so the three methods optimise the same objective.

In [ ]:
val_losses = {}
for C in C_GRID:
    m = LogisticRegression(C=C, max_iter=5000).fit(X_train, y_train)
    val_losses[C] = log_loss(y_val, m.predict_proba(X_val)[:, 1])
print({c: round(l, 4) for c, l in val_losses.items()})

BEST_C = min(val_losses, key=val_losses.get)
print("best C:", BEST_C)

sk_model = LogisticRegression(C=BEST_C, max_iter=5000).fit(X_train, y_train)
sk_val  = sk_model.predict_proba(X_val)[:, 1]
sk_test = sk_model.predict_proba(X_test)[:, 1]

## 11. Model 2: manual PyTorch loop

Same model, but we write each step ourselves:

1. **Forward:** `p = sigmoid(X @ w + b)`
2. **Loss:** binary cross-entropy plus an L2 penalty on `w`
3. **Backward:** `loss.backward()` computes the gradients
4. **Update:** `w -= LR * w.grad` (gradient descent), then reset the gradients

sklearn's `C` multiplies the sum of losses, so the equivalent penalty for a *mean* loss is `lam = 1 / (C * n)`.

In [ ]:
n_train, n_features = X_train.shape
lam = 1.0 / (BEST_C * n_train)

Xt = torch.tensor(X_train)
yt = torch.tensor(y_train, dtype=torch.float32)

w = torch.zeros(n_features, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

for epoch in range(EPOCHS):
    p = torch.sigmoid(Xt @ w + b)                                    # 1. forward
    bce = -(yt * torch.log(p + 1e-7) + (1 - yt) * torch.log(1 - p + 1e-7)).mean()
    loss = bce + 0.5 * lam * (w ** 2).sum()                           # 2. loss
    loss.backward()                                                   # 3. backward
    with torch.no_grad():                                             # 4. update
        w -= LR * w.grad
        b -= LR * b.grad
    w.grad.zero_()
    b.grad.zero_()
    if epoch % 1000 == 0:
        print(f"epoch {epoch:4d}  loss {loss.item():.4f}")

def manual_predict(X):
    with torch.no_grad():
        return torch.sigmoid(torch.tensor(X) @ w + b).numpy()

man_val, man_test = manual_predict(X_val), manual_predict(X_test)

## 12. Model 3: `torch.nn.Module` + `torch.optim`

Same maths, standard PyTorch tooling: `nn.Linear` holds `w` and `b`, `BCEWithLogitsLoss` applies the sigmoid and the cross-entropy in one stable step, and `optim.SGD` does the update. Weights start at zero to match the manual version, and we add the same L2 penalty.

In [ ]:
class LogisticModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)      # logits; sigmoid is applied outside

net = LogisticModel(n_features)
nn.init.zeros_(net.linear.weight)
nn.init.zeros_(net.linear.bias)

optimizer = torch.optim.SGD(net.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    loss = criterion(net(Xt), yt) + 0.5 * lam * (net.linear.weight ** 2).sum()
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0:
        print(f"epoch {epoch:4d}  loss {loss.item():.4f}")

net.eval()
def nn_predict(X):
    with torch.no_grad():
        return torch.sigmoid(net(torch.tensor(X))).numpy()

nn_val, nn_test = nn_predict(X_val), nn_predict(X_test)

## 13. Comparing the three methods

All three are evaluated on the same test set against the baseline. If the implementations are equivalent, their probabilities should be almost identical. Small differences are expected because sklearn's solver runs to convergence while our gradient descent stops after a fixed number of epochs.

In [ ]:
table = pd.DataFrame([
    baseline,
    evaluate("sklearn LogisticRegression", sk_val, sk_test),
    evaluate("PyTorch manual loop", man_val, man_test),
    evaluate("PyTorch nn.Module + optim", nn_val, nn_test),
])
print(table.to_string(index=False))

print("\nmax |probability difference| on the test set:")
print("  sklearn vs manual :", float(np.abs(sk_test - man_test).max()))
print("  sklearn vs nn     :", float(np.abs(sk_test - nn_test).max()))
print("  manual  vs nn     :", float(np.abs(man_test - nn_test).max()))

coefs = pd.DataFrame({
    "feature": FEATURES,
    "sklearn": sk_model.coef_[0],
    "manual": w.detach().numpy(),
    "nn_module": net.linear.weight.detach().numpy()[0],
}).round(3)
print("\ncoefficients (standardised features):")
print(coefs.to_string(index=False))

## 14. Save everything the app needs

Training is the only stage that trains anything. The Gradio app will **load** these files and never retrain:

- the three trained models,
- the preprocessing statistics (median, mean, std) and the feature list,
- the configuration and the results table,
- a table with every driver-race, its split, the true label and each model's predicted probability (for the plots and the threshold slider).

In [ ]:
joblib.dump(sk_model, ARTIFACTS / "sklearn_model.joblib")
torch.save({"w": w.detach(), "b": b.detach()}, ARTIFACTS / "manual_torch.pt")
torch.save(net.state_dict(), ARTIFACTS / "nn_module.pt")

preprocess = {
    "features": FEATURES,
    "median": {k: float(v) for k, v in median.items()},
    "mean": {k: float(v) for k, v in mean.items()},
    "std": {k: float(v) for k, v in std.items()},
}
(ARTIFACTS / "preprocess.json").write_text(json.dumps(preprocess, indent=2))

config = {
    "start_year": START_YEAR, "end_year": END_YEAR,
    "train_end": TRAIN_END, "val_end": VAL_END,
    "form_window": FORM_WINDOW, "reliability_window": RELIABILITY_WINDOW,
    "cost_fp": COST_FP, "cost_fn": COST_FN, "best_C": BEST_C,
    "epochs": EPOCHS, "lr": LR,
}
(ARTIFACTS / "config.json").write_text(json.dumps(config, indent=2))
table.to_json(ARTIFACTS / "metrics.json", orient="records", indent=2)

# predictions for every row of the modelling table
all_X = transform(data)
preds = data[["year", "round", "race_name", "driver", "team", "split", "scores_points"] + FEATURES].copy()
preds["baseline_pred"] = data["grid"].between(1, 10).astype(int).values
preds["p_sklearn"] = sk_model.predict_proba(all_X)[:, 1]
preds["p_manual"] = manual_predict(all_X)
preds["p_nn"] = nn_predict(all_X)
preds.to_csv(ARTIFACTS / "predictions.csv", index=False)

print("saved:", sorted(p.name for p in ARTIFACTS.iterdir()))

## Next

The Gradio app (`app.py` or `app.ipynb`) loads these artifacts and shows: the three-model comparison with a predicted-vs-actual plot, feature and target distributions, and a threshold slider that updates the confusion matrix and the expected cost.